# Setting

## Library

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Matplotlib global settings
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [3]:
# ML libraries
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

In [4]:
# Helper functions & model import
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *
from model import *

## Function

In [5]:
def select_uids_by_class(df, sample_number, class_col='Class', uid_col='uid', random_state=42):
    np.random.seed(random_state)
    uids_by_class = {}
    for c in df[class_col].unique():
        uids = df[df[class_col]==c][uid_col].unique()
        n_select = min(sample_number, len(uids))
        selected_uids = np.random.choice(uids, n_select, replace=False)
        uids_by_class[c] = set(selected_uids)
    return uids_by_class

In [6]:
def filter_by_selected_uids(df, uids_by_class, class_col='Class', uid_col='uid'):
    mask = np.zeros(len(df), dtype=bool)
    for c, uids in uids_by_class.items():
        mask |= ((df[class_col]==c) & (df[uid_col].isin(uids)))
    return df[mask]

In [7]:
from sklearn.model_selection import GroupShuffleSplit

def train_test_split_by_uid(df, test_size=0.2, class_col='Class', uid_col='uid', random_state=42):
    groups = df[uid_col]
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(df, df[class_col], groups))
    return df.iloc[train_idx], df.iloc[test_idx]

## Initial Setup

In [8]:
logtxt = ""

In [9]:
path_catboost = os.path.join(MODEL, "CatBoost_Fine_Tune")
path_xgboost = os.path.join(MODEL, "XGBoost_Fine_Tune")
path_lightgbm = os.path.join(MODEL, "LightGBM_Fine_Tune")

model_params_dict = {
    "Catboost": os.path.join(path_catboost, "best_params.yaml"),
    "XGBoost": os.path.join(path_xgboost, "best_params.yaml"),
    "LightGBM": os.path.join(path_lightgbm, "best_params.yaml"),
}

model_params_dict

{'Catboost': '/home/gpaek/SED-Classifier/notebook/../model/CatBoost_Fine_Tune/best_params.yaml',
 'XGBoost': '/home/gpaek/SED-Classifier/notebook/../model/XGBoost_Fine_Tune/best_params.yaml',
 'LightGBM': '/home/gpaek/SED-Classifier/notebook/../model/LightGBM_Fine_Tune/best_params.yaml'}

In [10]:
# Set experiment configs
test_name = "isolation_forest_base"
random_state = 42
test_size = 0.2
device_type = "cpu" # or gpu
n_jobs = 3
path_save = os.path.join(MODEL, test_name)
os.makedirs(path_save, exist_ok=True)

logtxt += "\nSet experiment configs\n"
logtxt += f"test_name: {test_name}\n"
logtxt += f"random_state: {random_state}\n"
logtxt += f"test_size: {test_size}\n"
logtxt += f"device_type: {device_type}\n"
logtxt += f"n_jobs: {n_jobs}\n"
logtxt += f"path_save: {path_save}\n"
logtxt += "\n"

- Source to Consider

In [11]:
sources_to_consider = [
	"AGN", 
	"Ia", 
	"II", 
	"Ibc", 
	"LBV", 
	"TDE", 
	"Nova", 
	"M dwarf", 
	"CV",
	"SLSN",
    #
    "KN"
]
logtxt += f"\nSources to consider: {sources_to_consider}\n"

In [12]:
# path_data = os.path.join(FEATURE_BALANCED_DATA, 'features_40.csv')
path_data = os.path.join(FEATURE_BALANCED_DATA, 'features_40_only_color.csv')
path_kn_data = os.path.join(FEATURE_ORIGINAL_DATA, 'features_40_kn.csv')

logtxt += f"\nBalanced Data Set\n"

In [13]:
path_save = os.path.join(MODEL, "three_top_models")
os.makedirs(path_save, exist_ok=True)

# Data

In [14]:
columns_to_use = list(data_dtype_dict.keys())

In [15]:
kn_data = pd.read_csv(
    path_kn_data,
    engine='c', 
    usecols=columns_to_use,
    dtype=data_dtype_dict,
)

In [16]:
original_data = pd.read_csv(
    path_data,
    engine='c', 
    usecols=columns_to_use,
    dtype=data_dtype_dict,
)

In [17]:
data = pd.concat([kn_data, original_data], ignore_index=True)

del kn_data
del original_data

In [18]:


data['uid'] = data['uid'].astype(str)
data['Class'] = data['Class'].astype(str)
print(f"Balanced Data: {len(data)}")

logtxt += f"Balanced Data: {len(data)}\n"

indx_type_to_consider = np.where(
	np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data = data.iloc[indx_type_to_consider[0]]

logtxt += f"Balanced: {len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}\n"
logtxt += "\n"


Balanced Data: 284962
11 sources to consider: 284962


- Training and Test Data

In [19]:

# - Split features/target
X_all = data.drop(columns=['Sample_ID', 'Class', 'uid'])
X_all.fillna(-99, inplace=True)
y_all = data['Class']
uids_all = data['uid']

#   KN
kn_idx = y_all == "KN"

X_kn = X_all[kn_idx]
X_kn.fillna(-99, inplace=True)
y_kn = y_all[kn_idx]
uids_kn = uids_all[kn_idx]

#
no_kn_idx = y_all != "KN"

X = X_all[no_kn_idx]
X.fillna(-99, inplace=True)
y = y_all[no_kn_idx]
uids = uids_all[no_kn_idx]

# - Split into train/test using GroupShuffleSplit by uid
gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
train_idx, test_idx = next(gss.split(X, y, groups=uids))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# - Label encode class for ML
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
class_names = np.array([str(c) for c in label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))])
print("Balanced: Class mapping:", class_names)

Balanced: Class mapping: ['AGN' 'CV' 'II' 'Ia' 'Ibc' 'LBV' 'M dwarf' 'Nova' 'SLSN' 'TDE']


In [20]:
from catboost import CatBoostClassifier, Pool
# CatBoost
# cat_train_data = Pool(data=X_train, label=y_train_encoded)
# cat_test_data = Pool(data=X_test, label=y_test_encoded)

import lightgbm as lgb
# lgb_train_data = lgb.Dataset(X_train, label=y_train_encoded)
# lgb_test_data = lgb.Dataset(X_test, label=y_test_encoded, reference=train_data)

import xgboost as xgb
# xgb_train_data = xgb.DMatrix(X_train, label=y_train_encoded)
# xgb_test_data = xgb.DMatrix(X_test, label=y_test_encoded)

In [21]:
del data

# Model

In [22]:
import time
from sklearn.metrics import mean_squared_error
from sklearn.metrics import f1_score
from xgboost.callback import EarlyStopping


from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [23]:
import numpy as np
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

class EnsembleVotingClassifier:
    def __init__(self, 
                 catboost_model_path,
                 xgboost_model_path,
                 lightgbm_model_path,
                 weights=None):
        """
        Args:
            catboost_model_path: path to CatBoost model (.cbm)
            xgboost_model_path: path to XGBoost model (.json or .bst)
            lightgbm_model_path: path to LightGBM model (.pkl)
            weights: list or tuple of weights [cat, xgb, lgb]; if None → soft voting
        """
        self.cat = CatBoostClassifier()
        self.cat.load_model(catboost_model_path)
        
        self.xgb = XGBClassifier()
        self.xgb.load_model(xgboost_model_path)
        
        self.lgb = joblib.load(lightgbm_model_path)

        if weights is not None:
            assert len(weights) == 3, "weights must have 3 elements"
            self.weights = np.array(weights) / np.sum(weights)
        else:
            self.weights = None

    def predict_proba(self, X):
        p_cat = self.cat.predict_proba(X)
        p_xgb = self.xgb.predict_proba(X)
        p_lgb = self.lgb.predict_proba(X)

        if self.weights is None:
            # Soft voting
            probs = (p_cat + p_xgb + p_lgb) / 3
        else:
            probs = (
                p_cat * self.weights[0] +
                p_xgb * self.weights[1] +
                p_lgb * self.weights[2]
            )
        return probs

    def predict(self, X):
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)

In [24]:
path_final_models = os.path.join(MODEL, "final_normal_class_model")

In [25]:
import json

path_weight = os.path.join(path_final_models, "ensemble_weights.json")
with open(path_weight, "r") as f:
    weight_data = json.load(f)
weights = weight_data["weights"]
print("Loaded weights:", weights)

Loaded weights: [0.12, 0.42, 0.46]


In [26]:
# 모델 로드 및 앙상블 분류기 생성
clf = EnsembleVotingClassifier(
    os.path.join(path_final_models, "catboost_model.cbm"),
    os.path.join(path_final_models, "xgboost_model.json"),
    os.path.join(path_final_models, "lightgbm_model.pkl"),
    weights=[0.2, 0.5, 0.3]  # None → soft voting
)

# 예측
y_pred = clf.predict(X_test)
y_probs = clf.predict_proba(X_test)

TBB Warning: The number of workers is currently limited to 9. The request for 20 workers is ignored. Further requests for more workers will be silently ignored until the limit changes.



In [27]:
n_jobs = -1

# Isolation Forest

In [28]:
import os, warnings, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20 
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')
warnings.filterwarnings("ignore")


In [29]:
import pandas as pd
import numpy as np
import re
import ast

In [ ]:
# Isolation Forest
import os, warnings, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import re
import ast

# ---------------------------------------------------------
# 1. Prepare Data for Isolation Forest
# ---------------------------------------------------------
# X_train already contains only non-KN samples from the train split
# X_test already contains only non-KN samples from the test split

# Combine the non-KN test data (normal class) and all KN data (anomaly class) for evaluation
# This creates a comprehensive test set to evaluate Isolation Forest's anomaly detection capabilities.
X_eval_normal = X_test # Non-KN samples from the test set
y_eval_normal = y_test # Corresponding labels

X_eval_anomaly = X_kn # All KN samples
y_eval_anomaly = y_kn # Corresponding labels

# Combine features and labels for the full evaluation set
X_eval = pd.concat([X_eval_normal, X_eval_anomaly], ignore_index=True)
y_eval_labels = pd.concat([y_eval_normal, y_eval_anomaly], ignore_index=True)

# Create binary true labels for evaluation: KN = 1 (anomaly), Others = 0 (normal)
y_eval_binary_true = (y_eval_labels == "KN").astype(int)

print(f"Isolation Forest Training Data Shape (non-KN): {X_train.shape}")
print(f"Isolation Forest Evaluation Data Shape (combined non-KN test + all KN): {X_eval.shape}")
print(f"Number of KN samples in evaluation set: {y_eval_binary_true.sum()}")

# ---------------------------------------------------------
# 2. Define and Train Isolation Forest Model
# ---------------------------------------------------------
# Initialize Isolation Forest model
# contamination='auto' is suitable as the training data (non-KN) has very low to zero contamination
model = IsolationForest(
    n_estimators=100,
    max_samples='auto',
    contamination='auto',
    random_state=random_state,
    n_jobs=n_jobs,
)

# Train the model exclusively on non-KN data (normal class)
print("\nTraining Isolation Forest model on non-KN data...")
model.fit(X_train)
print("Isolation Forest training complete.")

# Save the trained model
joblib.dump(model, f"{path_save}/isolation_forest_base.pkl")
print(f"Isolation Forest model saved to {path_save}/isolation_forest_base.pkl")

# ---------------------------------------------------------
# 3. Evaluate the Model
# ---------------------------------------------------------
print("\n[Evaluation on Training Set (non-KN)]")
# predict: normal=1, anomaly=-1
y_pred_train_raw = model.predict(X_train)
# Convert: -1 -> 1 (anomaly), 1 -> 0 (normal)
y_pred_train_bin = (y_pred_train_raw == -1).astype(int)
# True labels for training set (should be all normal, i.e., 0)
y_train_binary_true = (y_train == "KN").astype(int) # This should be all zeros if X_train contains no KN

print(classification_report(y_train_binary_true, y_pred_train_bin, target_names=["Normal", "Anomaly"]))


print("\n[Evaluation on Combined Test Set (non-KN test + all KN)]")
# Get raw predictions (-1 for anomaly, 1 for normal)
y_pred_eval_raw = model.predict(X_eval)
# Convert to binary: -1 -> 1 (anomaly), 1 -> 0 (normal)
y_pred_eval_bin = (y_pred_eval_raw == -1).astype(int)

# Classification Report
print(classification_report(y_eval_binary_true, y_pred_eval_bin, target_names=["Normal", "Anomaly"]))

# ---------------------------------------------------------
# 4. Confusion Matrix Visualization
# ---------------------------------------------------------
def plot_confusion_matrix(y_true, y_pred, title, savepath):
    conf_matrix = confusion_matrix(y_true, y_pred)
    # Calculate percentages for better readability
    conf_matrix_percent = conf_matrix / conf_matrix.sum(axis=1, keepdims=True) * 100

    plt.figure(figsize=(6, 5))
    ax = sns.heatmap(conf_matrix_percent, annot=True, fmt=".1f", cmap="Blues",
                     xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"],
                     vmin=0, vmax=100, cbar_kws={'label': '[%]'})
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(savepath)
    plt.show() # Display the plot after saving

plot_confusion_matrix(y_eval_binary_true, y_pred_eval_bin,
                      title="Isolation Forest - Combined Evaluation Set",
                      savepath=f"{path_save}/confusion_matrix_iforest_combined_eval.png")

# ---------------------------------------------------------
# 5. Anomaly Score Distribution and ROC Curve
# ---------------------------------------------------------
# Calculate anomaly scores for the evaluation set
# Lower score indicates higher anomaly likelihood for IsolationForest
anomaly_scores_eval = -model.decision_function(X_eval) # Negate for typical 'higher score = more anomalous' interpretation

# Plotting anomaly score distribution
plt.figure(figsize=(10, 6))
plt.hist(anomaly_scores_eval[y_eval_binary_true == 1], bins=50, alpha=0.7, label='KN (Anomaly)', histtype='stepfilled', density=True, color='red')
plt.hist(anomaly_scores_eval[y_eval_binary_true == 0], bins=50, alpha=0.7, label='Non-KN (Normal)', histtype='stepfilled', density=True, color='blue')
plt.xlabel("Anomaly Score (Higher = More Anomalous)")
plt.ylabel("Density")
plt.title("Distribution of Isolation Forest Anomaly Scores on Evaluation Set")
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.savefig(f"{path_save}/anomaly_score_distribution.png")
plt.show()

# ROC Curve for anomaly detection
fpr, tpr, thresholds = roc_curve(y_eval_binary_true, anomaly_scores_eval)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve for Anomaly Detection')
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig(f"{path_save}/roc_curve_iforest.png")
plt.show()

# ---------------------------------------------------------
# 6. Evaluation with a Custom Threshold (Optional)
# ---------------------------------------------------------
# You can select a threshold based on the ROC curve or score distribution
# For example, to optimize for a specific precision/recall trade-off
optimal_threshold = np.percentile(anomaly_scores_eval[y_eval_binary_true == 0], 95) # Example: threshold where 5% of normal samples are classified as anomalies
print(f"\nExample: Custom threshold set at 95th percentile of normal scores: {optimal_threshold:.4f}")

# Apply the custom threshold to make binary predictions
y_pred_eval_bin_custom_thresh = (anomaly_scores_eval > optimal_threshold).astype(int)

print(f"\n[Evaluation with Custom Threshold ({optimal_threshold:.4f})]")
print(classification_report(y_eval_binary_true, y_pred_eval_bin_custom_thresh, target_names=["Normal", "Anomaly"]))

# Confusion Matrix for custom threshold
plot_confusion_matrix(y_eval_binary_true, y_pred_eval_bin_custom_thresh,
                      title=f"Isolation Forest - Custom Threshold ({optimal_threshold:.2f})",
                      savepath=f"{path_save}/confusion_matrix_iforest_custom_thresh.png")

Isolation Forest Training Data Shape (non-KN): (227957, 780)
Isolation Forest Evaluation Data Shape (combined non-KN test + all KN): (57005, 780)
Number of KN samples in evaluation set: 2209

Training Isolation Forest model on non-KN data...
Isolation Forest training complete.
Isolation Forest model saved to /home/gpaek/SED-Classifier/notebook/../model/three_top_models/isolation_forest_base.pkl

[Evaluation on Training Set (non-KN)]
              precision    recall  f1-score   support

      Normal       1.00      0.93      0.96    227957
     Anomaly       0.00      0.00      0.00         0

    accuracy                           0.93    227957
   macro avg       0.50      0.46      0.48    227957
weighted avg       1.00      0.93      0.96    227957


[Evaluation on Combined Test Set (non-KN test + all KN)]
              precision    recall  f1-score   support

      Normal       0.97      0.93      0.95     54796
     Anomaly       0.16      0.35      0.22      2209

    accuracy  


KeyboardInterrupt



Error in callback <function _draw_all_if_interactive at 0x15200c9fef70> (for post_execute):
